In [1]:
import os
from pyspark.sql import SparkSession

# Khởi tạo Spark Session cho Lab 4
spark = (SparkSession.builder
    .appName("Lab4_SparkML_Home")
    # Nạp gói Kafka (bắt buộc để làm Ex 0)
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1")
    # Kích hoạt AQE theo đặc tả trang 2 của Lab 4
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .master("local[*]") 
    .getOrCreate())

print("--- Spark Session cho Lab 4 đã sẵn sàng! ---")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/14 07:49:11 WARN Utils: Your hostname, HungThinh, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/14 07:49:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/thinh/BigData/bigdata_lab/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/thinh/.ivy2.5.2/cache
The jars for the packages stored in: /home/thinh/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5efdc8a3-074e-4fd9-a356-b34ff0519568;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	foun

--- Spark Session cho Lab 4 đã sẵn sàng! ---


In [3]:
from confluent_kafka.admin import AdminClient, NewTopic

admin_client = AdminClient({'bootstrap.servers': 'localhost:9092,localhost:9192,localhost:9292'})
topic_names = ['Lab1_movies', 'Lab1_ratings', 'Lab1_tags']
admin_client.delete_topics(topic_names, operation_timeout=10)
new_topics = [NewTopic(topic=name, num_partitions=1, replication_factor=1) for name in topic_names]
fs = admin_client.create_topics(new_topics)

for topic, f in fs.items():
    try:
        f.result()
        print(f"Topic '{topic}' đã được khởi tạo thành công!")
    except Exception as e:
        print(f"Bỏ qua/Lỗi tại {topic}: {e}")

Topic 'Lab1_movies' đã được khởi tạo thành công!
Topic 'Lab1_ratings' đã được khởi tạo thành công!
Topic 'Lab1_tags' đã được khởi tạo thành công!


In [4]:
import kagglehub
from pyspark.sql.functions import to_json, struct

# 1. Tải dataset từ Kaggle
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

# 2. Đọc file CSV bằng Spark 
df_r = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_m = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_t = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

# 3. Hàm đẩy dữ liệu vào Kafka
def push_to_kafka(df, topic):
    df.selectExpr("to_json(struct(*)) AS value") \
      .write.format("kafka") \
      .option("kafka.bootstrap.servers", "localhost:9092,localhost:9192,localhost:9292") \
      .option("topic", topic) \
      .save()
    print(f"Đã nạp dữ liệu cho {topic}")

# Thực hiện nạp cho 3 topic 
push_to_kafka(df_m, "Lab1_movies")
push_to_kafka(df_r, "Lab1_ratings")
push_to_kafka(df_t, "Lab1_tags")

/home/thinh/BigData/bigdata_lab/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đã nạp dữ liệu cho Lab1_movies


Đã nạp dữ liệu cho Lab1_ratings
Đã nạp dữ liệu cho Lab1_tags


In [12]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col

# 1. Định nghĩa Schema cho movies, ratings và tags theo đặc tả Lab 4
movie_schema = StructType([
    StructField("movieId", IntegerType()), 
    StructField("title", StringType()), 
    StructField("genres", StringType())
])
rating_schema = StructType([
    StructField("userId", IntegerType()), 
    StructField("movieId", IntegerType()), 
    StructField("rating", DoubleType()), 
    StructField("timestamp", LongType())
])
tag_schema = StructType([
    StructField("userId", IntegerType()), 
    StructField("movieId", IntegerType()), 
    StructField("tag", StringType()), 
    StructField("timestamp", LongType())
])

# 2. Hàm load dữ liệu từ Kafka vào DataFrame
def load_kafka(topic, schema):
    return (spark.read.format("kafka")
        .option("kafka.bootstrap.servers", "localhost:9092,localhost:9192,localhost:9292")
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .load()
        .select(from_json(col("value").cast("string"), schema).alias("data"))
        .select("data.*"))

# 3. Chuyển đổi dữ liệu sang định dạng chuẩn
df_movies = load_kafka("Lab1_movies", movie_schema)
df_ratings = load_kafka("Lab1_ratings", rating_schema)
df_tags = load_kafka("Lab1_tags", tag_schema)

print("--- Exercise 0: Hoàn thành! ---")
df_movies.show(5)
df_ratings.show(5)
df_tags.show(5)

--- Exercise 0: Hoàn thành! ---
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows
+------+-------+---------------+----------+
|userId|movieId|            tag| timestamp|
+------+-------+---------------+----------+
|     2|  6075

Exercise1

In [21]:
from pyspark.sql.functions import avg, count, when, col, concat_ws, collect_list, split

# 1. Giữ nguyên nhãn >= 4 theo đúng đề bài
df_labeled = df_ratings.withColumn("label", when(col("rating") >= 4, 1).otherwise(0))

# 2. Numerics: Thống kê User và Movie
movie_stats = df_ratings.groupBy("movieId").agg(
    avg("rating").alias("movie_avg_rating"),
    count("rating").alias("movie_rating_count")
)
user_stats = df_ratings.groupBy("userId").agg(
    avg("rating").alias("user_avg_rating"),
    count("rating").alias("user_rating_count")
)

# 3. Text: Kết hợp Title và Tags
user_movie_tags = df_tags.groupBy("userId", "movieId").agg(
    concat_ws(" ", collect_list("tag")).alias("user_tags")
)

# 4. Join và xử lý Categorical multi-label
train_df = df_labeled.join(df_movies, "movieId") \
    .join(movie_stats, "movieId") \
    .join(user_stats, "userId") \
    .join(user_movie_tags, ["userId", "movieId"], "left") \
    .fillna("")

# Tạo cột genres_tokens bằng cách split chuỗi "|"
train_df = train_df.withColumn("genres_tokens", split(col("genres"), "\\|"))
# Kết hợp Title và Tags thành cột text để xử lý văn bản
train_df = train_df.withColumn("text_metadata", concat_ws(" ", col("title"), col("user_tags")))

print("Cấu trúc dữ liệu cho huấn luyện:")
train_df.select("text_metadata", "genres_tokens", "label").show(5, truncate=False)

Cấu trúc dữ liệu cho huấn luyện:


+---------------------------------+------------------------------------+-----+
|text_metadata                    |genres_tokens                       |label|
+---------------------------------+------------------------------------+-----+
|Men in Black (a.k.a. MIB) (1997) |[Action, Comedy, Sci-Fi]            |0    |
|King Kong (1933)                 |[Action, Adventure, Fantasy, Horror]|1    |
|Men in Black (a.k.a. MIB) (1997) |[Action, Comedy, Sci-Fi]            |0    |
|Galaxy Quest (1999)              |[Adventure, Comedy, Sci-Fi]         |0    |
|Dirty Dancing (1987)             |[Drama, Musical, Romance]           |0    |
+---------------------------------+------------------------------------+-----+
only showing top 5 rows


In [22]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, CountVectorizer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression

# Bước 1: Xử lý Text (Title + Tags) - Dùng CountVectorizer thay vì HashingTF
tokenizer = Tokenizer(inputCol="text_metadata", outputCol="words")
cv_text = CountVectorizer(inputCol="words", outputCol="text_features")

# Bước 2: Xử lý Categorical multi-label (genres_tokens)
cv_genres = CountVectorizer(inputCol="genres_tokens", outputCol="genre_features")

# Bước 3: Gom nhóm đặc trưng (Numerics gồm movie_avg_rating và user_avg_rating)
assembler = VectorAssembler(
    inputCols=["text_features", "genre_features", "movie_avg_rating", "user_avg_rating"],
    outputCol="raw_features"
)

# Bước 4: Chuẩn hóa và Mô hình
scaler = StandardScaler(inputCol="raw_features", outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

# Tạo Pipeline mới
pipeline = Pipeline(stages=[tokenizer, cv_text, cv_genres, assembler, scaler, lr])

# Chia dữ liệu và Huấn luyện
train_data, test_data = train_df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train_data)
print("--- Đã huấn luyện xong mô hình! ---")

26/05/14 08:34:00 WARN DAGScheduler: Broadcasting large task binary with size 1050.9 KiB
26/05/14 08:34:02 WARN DAGScheduler: Broadcasting large task binary with size 1051.5 KiB
26/05/14 08:34:03 WARN DAGScheduler: Broadcasting large task binary with size 1051.5 KiB
26/05/14 08:34:03 WARN DAGScheduler: Broadcasting large task binary with size 1051.5 KiB
26/05/14 08:34:04 WARN DAGScheduler: Broadcasting large task binary with size 1051.5 KiB
26/05/14 08:34:04 WARN DAGScheduler: Broadcasting large task binary with size 1051.5 KiB
26/05/14 08:34:04 WARN DAGScheduler: Broadcasting large task binary with size 1051.5 KiB
26/05/14 08:34:04 WARN DAGScheduler: Broadcasting large task binary with size 1051.5 KiB
26/05/14 08:34:04 WARN DAGScheduler: Broadcasting large task binary with size 1051.5 KiB
26/05/14 08:34:04 WARN DAGScheduler: Broadcasting large task binary with size 1051.5 KiB
26/05/14 08:34:04 WARN DAGScheduler: Broadcasting large task binary with size 1051.5 KiB
26/05/14 08:34:04 WAR

--- Đã huấn luyện xong mô hình! ---


In [23]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Dự đoán trên tập Test
predictions = model.transform(test_data)

# 1. Tính AUC
evaluator_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
auc = evaluator_auc.evaluate(predictions)

# 2. Tính F1-Score
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
f1 = evaluator_f1.evaluate(predictions)

print(f"--- Kết quả đánh giá ---")
print(f"AUC (Primary): {auc:.4f}")
print(f"F1 Score: {f1:.4f}")

# 3. Tạo Ma trận nhầm lẫn (Confusion Matrix) 2x2
print("\nMa trận nhầm lẫn (Confusion Matrix):")
predictions.groupBy("label", "prediction").count().show()

26/05/14 08:35:15 WARN DAGScheduler: Broadcasting large task binary with size 1144.4 KiB
26/05/14 08:35:22 WARN DAGScheduler: Broadcasting large task binary with size 1156.3 KiB


--- Kết quả đánh giá ---
AUC (Primary): 0.7764
F1 Score: 0.7104

Ma trận nhầm lẫn (Confusion Matrix):


26/05/14 08:35:27 WARN DAGScheduler: Broadcasting large task binary with size 1153.1 KiB


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0| 2839|
|    0|       0.0| 7489|
|    1|       1.0| 6900|
|    0|       1.0| 3030|
+-----+----------+-----+



26/05/14 08:35:28 WARN DAGScheduler: Broadcasting large task binary with size 1134.1 KiB


In [24]:
import numpy as np

# Lấy các model thành phần từ PipelineModel
lr_model = model.stages[-1]
vocab_text = model.stages[1].vocabulary
vocab_genres = model.stages[2].vocabulary

# Tạo danh sách tên đặc trưng theo đúng thứ tự trong VectorAssembler
# Lưu ý: 2 cột cuối là movie_avg_rating và user_avg_rating
feature_names = vocab_text + vocab_genres + ["movie_avg_rating", "user_avg_rating"]

# Lấy trọng số mô hình
weights = lr_model.coefficients.toArray()
indices = np.argsort(weights)

print("--- Top 10 Positive Feature Signals (Tín hiệu tích cực) ---")
for i in indices[-10:][::-1]:
    if i < len(feature_names):
        print(f"{feature_names[i]:<20} : {weights[i]:.4f}")

print("\n--- Top 10 Negative Feature Signals (Tín hiệu tiêu cực) ---")
for i in indices[:10]:
    if i < len(feature_names):
        print(f"{feature_names[i]:<20} : {weights[i]:.4f}")


--- Top 10 Positive Feature Signals (Tín hiệu tích cực) ---
user_avg_rating      : 0.9561
movie_avg_rating     : 0.7747
Documentary          : 0.0562
yojimbo              : 0.0458
sexuality            : 0.0432
humor                : 0.0429
cinematography       : 0.0421
clever               : 0.0400
silly                : 0.0390
ending               : 0.0379

--- Top 10 Negative Feature Signals (Tín hiệu tiêu cực) ---
godzilla             : -0.0470
clockers             : -0.0464
knowing              : -0.0410
2,                   : -0.0409
airheads             : -0.0406
kindergarten         : -0.0406
british              : -0.0401
haunted              : -0.0387
twins                : -0.0383
shadow,              : -0.0373
